# Data audit — Polish companies bankruptcy (`5year.arff`)

**Source:** UCI ML Repository, dataset 365 — Polish Companies Bankruptcy Data.

**Horizon.** File numbering is counter-intuitive: `Nyear.arff` holds financial ratios from year *N*
of the observation window and a bankruptcy label for *(5 − N)* years ahead. `5year.arff` is
therefore a **1-year-ahead** forecast — the strongest signal and the highest positive rate of the
five files.

**Purpose.** Verify that the data matches its documentation before any analysis, and record the
decisions that feed into preprocessing and feature engineering.

**Headline findings**

1. The file is **pre-filtered**: companies with zero sales were removed before publication.
2. Class imbalance is severe — **7.0%** positives.
3. Missingness is **structural, not accidental** — it encodes the financial state of a company and
   is related to the target.

**Note on the frame used below.** `load_data` drops full-row duplicates at load time, so every
number in this notebook refers to the deduplicated frame of 5850 rows. Section 9 reloads the raw
file to document what was removed and why.

In [1]:
import numpy as np
import pandas as pd

from src.config import DATA_5YEAR_PATH, TARGET_COL
from src.data import load_data

df = load_data(DATA_5YEAR_PATH)

## 1. Schema check

Does the data match its documented description: 64 numeric features and one binary target?

In [2]:
print("shape:", df.shape)
print(df.dtypes.value_counts())
print("target values:", sorted(df[TARGET_COL].unique()))

shape: (5850, 65)
float64    64
int64       1
Name: count, dtype: int64
target values: [np.int64(0), np.int64(1)]


The ARFF header declares 64 numeric attributes and a single nominal target, `class`, whose values
are stored as byte strings; `load_data` decodes it to `int` and renames the column to
`is_bankrupt`.

Feature columns keep their original `Attr1..Attr64` names. Human-readable descriptions live in
`config.FEATURE_LABELS` and are used for plot labels only — renaming the columns themselves would
sever the link to the UCI attribute description and to the published literature on this dataset,
which refers to features by number.

The raw file holds 5910 rows; the frame above is smaller by the 60 duplicates removed at load time
(section 9).

## 2. Provenance: the data was pre-filtered

In [3]:
# The @relation header is truncated by scipy's parser, so read it from the file directly.
with open(DATA_5YEAR_PATH, encoding="utf-8") as f:
    print(f.readline().strip())

@relation '5year-weka.filters.unsupervised.instance.SubsetByExpression-Enot ismissing(ATT20)'


The header records a Weka filter: `SubsetByExpression -E not ismissing(ATT20)`. Rows where `Attr20`
could not be computed were dropped before publication. Since `Attr20 = (inventory × 365) / sales`,
the ratio is undefined exactly when sales are zero — so the removed rows are **companies with no
revenue over the period**. The authors do not document why this criterion was chosen; the mechanism
is clear, the intent is not.

This is a **selection bias**: the sample was filtered on a property correlated with the outcome.
Companies with zero revenue — newly registered, dormant, or already ceasing operations — are more
likely to fail than average. Removing a group enriched in bankruptcies **deflates** the observed
positive rate, so the 7.0% below is lower than the true rate in the population of Polish companies.

**Consequence for deployment:** the model is valid only for companies with non-zero revenue.
Applied to a dormant firm it will still return a probability, but nothing supports that number.
This goes in the README Limitations section.

## 3. Class balance

In [4]:
print(df[TARGET_COL].value_counts())
print(df[TARGET_COL].value_counts(normalize=True))

is_bankrupt
0    5442
1     408
Name: count, dtype: int64
is_bankrupt
0    0.930256
1    0.069744
Name: proportion, dtype: float64


408 positives out of 5850 — a positive rate of **0.070**.

This number is the **PR-AUC baseline**: a classifier assigning random scores achieves a PR-AUC
approximately equal to the positive rate, because any randomly selected subset contains the same
share of positives as the full sample. Every result later in the project is therefore reported as a
*lift over 0.070*, not as an absolute value.

**Accuracy is not used anywhere in this project.** A model that labels every company as healthy is
wrong 408 times and right 5442 times — 93.0% accuracy while catching no bankruptcies at all.
Accuracy weights errors by class frequency, but the rare class is the one that matters here, and a
missed bankruptcy and a false alarm do not carry the same cost. The metrics are PR-AUC and
precision@top-k.

## 4. Missing values

In [5]:
na_rate = df.isna().mean().sort_values(ascending=False)

print(na_rate.head(10))
print("Attr20 missing rate:", na_rate["Attr20"])
print("rows with any NaN:", df.isna().any(axis=1).mean())
print(
    "rows with any NaN, excluding Attr37:",
    df.drop(columns="Attr37").isna().any(axis=1).mean(),
)

Attr37    0.431624
Attr27    0.066667
Attr45    0.045470
Attr60    0.045470
Attr24    0.023077
Attr64    0.018120
Attr53    0.018120
Attr28    0.018120
Attr54    0.018120
Attr21    0.017607
dtype: float64
Attr20 missing rate: 0.0
rows with any NaN: 0.488034188034188
rows with any NaN, excluding Attr37: 0.1552136752136752


Missingness is dominated by a single column: `Attr37` at **43.2%**, six times the next one
(`Attr27`, 6.7%), with everything from the fifth position down below 2.4%. `Attr20` is at exactly
**0.0%**, confirming the filter above.

**Listwise deletion is not an option.** At least one feature is missing in **48.8%** of rows;
dropping them would destroy half the sample and roughly 200 of the 408 positives. Even excluding
`Attr37`, **15.5%** of rows are still affected — missingness is spread thinly across many features,
not confined to one column.

Crucially, a missing value here does **not** mean incomplete data collection. Every feature is a
ratio, and a missing value marks an undefined denominator — a real financial state of the company.
The handling therefore cannot be a blanket fill.

## 5. Is missingness in `Attr37` related to the target?

`Attr37 = (current assets − inventories) / long-term liabilities`. The denominator is zero for
companies carrying **no long-term debt**, which makes the ratio undefined.

In [6]:
mask = df["Attr37"].isna()
print(df.groupby(mask)[TARGET_COL].agg(["size", "sum", "mean"]))

        size  sum      mean
Attr37                     
False   3325  199  0.059850
True    2525  209  0.082772


The bankruptcy rate is **8.3%** among companies with `Attr37` missing versus **6.0%** among the
rest — a factor of 1.38. Both groups are large (2525 and 3325 rows, 209 and 199 positives), so the
gap is far outside sampling noise (two-proportion z ≈ 3.4). **The fact of the value being missing
carries signal in itself.**

A plausible economic reading: long-term credit has to be granted. A bank does not lend long to a
firm it considers risky, so the absence of long-term debt in an operating company can indicate lack
of access to credit rather than financial discipline — such a firm survives on short-term
obligations that must be rolled over continuously.

**Consequences:**

- **Linear branch:** imputation must be paired with a missing indicator
  (`SimpleImputer(add_indicator=True)`). Filling with the median alone erases the signal; filling
  with 0 or −1 is worse still, since it places "no debt at all" on the same numeric scale as real
  coverage ratios and forces one shared weight onto two different meanings.
- **Boosting branch:** leave NaN untouched. LightGBM learns a routing direction for missing values
  at every split, so it extracts this signal natively.

This is an **association, not a causal claim**. The elevated risk may come from the type of company
that tends to have no long-term debt rather than from the absence of debt itself. Sufficient for
prediction, not for causal conclusions — the same caveat applies to the SHAP results later on.

## 6. Infinities

All features are ratios, so division by zero is possible. `isna()` does not catch `inf`: it is a
valid float that passes every missing-value check, then silently breaks scalers and linear models.

In [7]:
# np.isinf raises on non-numeric columns, so the target is excluded.
inf_mask = np.isinf(df.drop(columns=TARGET_COL))

print("total inf values:", inf_mask.sum().sum())
print("affected rows:", inf_mask.any(axis=1).sum())

total inf values: 0
affected rows: 0


No infinities. Undefined ratios were encoded as missing values during data preparation; whether
`inf` appeared at an intermediate step cannot be recovered from the file, and ARFF has no syntax
for it in any case.

## 7. Constant and empty columns

`nunique()` ignores NaN: an all-NaN column returns 0, a true constant returns 1. Both are useless
to a model.

In [8]:
print(df.drop(columns=TARGET_COL).nunique().sort_values().head())

Attr59    3254
Attr37    3260
Attr6     3539
Attr21    4420
Attr9     4823
dtype: int64


Nothing to drop — the minimum is 3254 distinct values. The two lowest counts are explained rather
than anomalous: `Attr37` is missing in 2525 rows (excluded from the count), and `Attr59` has a large
spike at zero, examined next.

## 8. The same state encoded twice: `Attr37` missing and `Attr59 == 0`

`Attr59 = long-term liabilities / equity` places the same quantity in the **numerator**, whereas
`Attr37` has it in the denominator. Zero long-term debt therefore produces NaN in one feature and a
legitimate 0.0 in the other.

In [9]:
print(df["Attr59"].value_counts().head())
print("agreement:", (df["Attr37"].isna() == (df["Attr59"] == 0)).mean())

mismatch = df[df["Attr37"].isna() != (df["Attr59"] == 0)]
print(mismatch[["Attr37", "Attr59"]])

Attr59
0.000000    2524
0.014242       2
0.002256       2
0.150150       2
0.168100       2
Name: count, dtype: int64
agreement: 0.9991452991452991
      Attr37  Attr59
4533     0.0     0.0
4718     0.0     0.0
4795     NaN     NaN
4827     NaN     NaN
5820     NaN     NaN


The two encodings agree on **99.9%** of rows — 2524 zeros in `Attr59` against 2525 missing values in
`Attr37`. The five mismatches split into two distinct cases:

- **Three rows** where both features are NaN. These companies have zero equity as well, so `Attr59`
  breaks on its own denominator.
- **Two rows** where `Attr37` is 0.0 rather than NaN, so long-term debt does exist and the zero comes
  from the numerator (`current assets − inventories = 0`). `Attr59` is presumably rounded to zero at
  the dataset's stored precision.

**Implication for feature engineering.** The planned `has_no_long_term_debt` flag may be redundant
for LightGBM: the same partition is already reachable through a threshold on `Attr59`. For the linear
branch it is not redundant — there, `Attr59 = 0` sits on a continuous scale next to genuinely small
values and cannot be isolated as a category. Decided on CV with std later, not here.

## 9. Duplicate rows

The working frame is already deduplicated, so this section reloads the raw file to document what was
removed. Two questions: are the duplicates genuine repeats or an artifact of `NaN == NaN` matching,
and do identical feature vectors ever carry conflicting labels?

In [10]:
raw = load_data(DATA_5YEAR_PATH, deduplicate=False)

# Same keep= setting on both sides, otherwise the counts are not comparable.
print("raw shape:", raw.shape)
print("duplicates incl. target:", raw.duplicated().sum())
print("duplicates on features only:", raw.drop(columns=TARGET_COL).duplicated().sum())

# keep=False marks every row of a duplicated group, not just the extra ones.
dups = raw[raw.duplicated(keep=False)]

print(dups.notna().sum(axis=1).describe())
print(dups[TARGET_COL].value_counts())

raw shape: (5910, 65)
duplicates incl. target: 60
duplicates on features only: 60
count    120.000000
mean      64.466667
std        0.888142
min       60.000000
25%       64.000000
50%       65.000000
75%       65.000000
max       65.000000
dtype: float64
is_bankrupt
0    116
1      4
Name: count, dtype: int64


**Decision: drop duplicates at load time, keeping the first occurrence of each group.** Three
findings support it:

1. **They are genuine repeats, not NaN artifacts.** Among the rows involved, the least populated has
   60 of 65 values filled and the median has all 65. Sixty float ratios matching to the last digit
   cannot happen by chance between two independent companies.
2. **Labels are consistent.** Duplicate counts with and without the target column are identical, so
   no pair carries contradictory labels.
3. **The cost is negligible.** Only 4 of the affected rows are positives, so deduplication removes at
   most 2 bankruptcies — under half a percent of the signal.

**What this prevents:** with copies of one company split across train and test, the model memorises
the object instead of the financial pattern, and the metric is inflated — a form of data leakage.

Deduplication is not a learned transformation (it estimates nothing from the data), so it belongs in
`data.py` at load time rather than inside the `Pipeline`.

## 10. Near-duplicate rows

`duplicated()` matches byte-identical rows only. A company entered twice with one recomputed figure
survives deduplication, and a row-wise split can then place its copies on both sides — the model
memorises the object instead of the pattern. The file carries no company identifier, so `GroupKFold`
is not available and the risk has to be measured rather than engineered away.

**Method.** Round all features to *k* decimals and re-run `duplicated()`. Rounding imposes a grid on
the feature space and treats one cell as a match. A range of *k* is tabulated rather than one value
chosen: the shape of the curve is the signal — a flat zero followed by a jump means a cluster of
near-copies, smooth growth from the first step means ordinary data density.

**Criterion, fixed before running the code.** PR-AUC is driven by the ranking of the ~82 positives in
a validation fold; one memorised negative is diluted among more than a thousand. Taking 1/82 ≈ 0.012
as the upper bound per leaked positive and 0.8 as the probability that a pair splits across folds in
5-fold CV, the validation scheme needs to change only at the order of **45 positive pairs** against a
between-fold std of 0.02–0.05. Below that the finding is a documented limitation, not a design
change. The threshold is stated for CV rather than the holdout: CV carries every model comparison in
the project and its split probability is the harsher of the two (0.8 against 0.32).

In [11]:
FEATURE_COLS = [x for x in df.columns if x != TARGET_COL]


def count_near_duplicates(frame: pd.DataFrame, k: int) -> dict[str, int]:
    """Count duplicate groups and affected rows after rounding to k decimals."""
    rounded = frame.round(k)
    mask = rounded.duplicated(keep=False)
    return {
        # keep=False marks every member of a group, so this counts rows, not extra copies.
        "rows": int(mask.sum()),
        # One representative survives per group, so the length is the number of groups.
        "groups": len(rounded[mask].drop_duplicates()),
    }


results = []

# Both variants: a pair matching on features but not on the label would argue against
# the "one company recorded twice" reading.
for k in range(1, 7):
    features_only = count_near_duplicates(df[FEATURE_COLS], k)
    with_target = count_near_duplicates(df, k)
    results.append(
        {
            "k": k,
            "groups_features": features_only["groups"],
            "rows_features": features_only["rows"],
            "groups_with_target": with_target["groups"],
            "rows_with_target": with_target["rows"],
        }
    )

near_dup_summary = pd.DataFrame(results)
print(near_dup_summary)

   k  groups_features  rows_features  groups_with_target  rows_with_target
0  1                5             10                   5                10
1  2                2              4                   2                 4
2  3                1              2                   1                 2
3  4                1              2                   1                 2
4  5                1              2                   1                 2
5  6                0              0                   0                 0


In [12]:
# The mask is built on the rounded frame but applied to the original one:
# the point is to see what the two rows actually differ in.
mask = df.round(5).duplicated(keep=False)

pair = df[mask].T
pair["diff"] = pair.iloc[:, 1] - pair.iloc[:, 0]
print(pair.to_string())

                     3280          4462      diff
Attr1            0.005535      0.005535  0.000000
Attr2            0.792090      0.792090  0.000000
Attr3            0.075540      0.075540  0.000000
Attr4            1.167900      1.167900  0.000000
Attr5         -277.840000   -277.840000  0.000000
Attr6            0.000000      0.000000  0.000000
Attr7            0.003254      0.003254  0.000000
Attr8            0.262480      0.262480  0.000000
Attr9            0.706650      0.706650  0.000000
Attr10           0.207910      0.207910  0.000000
Attr11           0.052946      0.052946  0.000000
Attr12           0.007231      0.007231  0.000000
Attr13           0.029778      0.029778  0.000000
Attr14           0.003254      0.003254  0.000000
Attr15       13740.000000  13740.000000  0.000000
Attr16           0.026565      0.026565  0.000000
Attr17           1.262500      1.262500  0.000000
Attr18           0.003254      0.003254  0.000000
Attr19           0.004605      0.004605  0.000000


In [13]:
K = 1

rounded = df.round(K)
mask = rounded.duplicated(keep=False)

print("rows flagged:", int(mask.sum()))
print("positives among them:", int(df.loc[mask, TARGET_COL].sum()))

# Group by the full rounded signature; dropna=False keeps groups whose key contains NaN,
# which Attr37 alone would otherwise remove from 43% of the frame.
groups = rounded[mask].groupby(list(rounded.columns), dropna=False).groups

for i, idx in enumerate(groups.values(), start=1):
    pair = df.loc[idx].T
    pair["diff"] = pair.iloc[:, 1] - pair.iloc[:, 0]
    # NaN - NaN is NaN and NaN != 0 is True, so a feature missing in both rows
    # would otherwise be reported as a difference.
    differs = pair["diff"].fillna(0) != 0
    print(f"\n--- group {i}: rows {list(idx)}, features differing: {differs.sum()} ---")
    print(pair[differs].to_string())

rows flagged: 10
positives among them: 0

--- group 1: rows [3288, 5435], features differing: 1 ---
           3288     5435    diff
Attr36  0.49149  0.50719  0.0157

--- group 2: rows [2859, 5354], features differing: 1 ---
          2859    5354    diff
Attr36  4.6379  4.6394  0.0015

--- group 3: rows [3280, 4462], features differing: 1 ---
            3280     4462      diff
Attr57  0.026624  0.02662 -0.000004

--- group 4: rows [521, 5404], features differing: 6 ---
              521       5404      diff
Attr5   32.102000  32.051000 -0.051000
Attr33  19.765000  19.795000  0.030000
Attr34  19.283000  19.312000  0.029000
Attr36   3.360200   3.364300  0.004100
Attr47  49.845000  49.769000 -0.076000
Attr52   0.050594   0.050517 -0.000077

--- group 5: rows [1957, 5425], features differing: 2 ---
          1957    5425    diff
Attr36  2.3954  2.4080  0.0126
Attr47  1.3148  1.3147 -0.0001


**Five pairs, none of them positive.** Counts with and without the target are identical, so no pair
carries conflicting labels, and `rows` is exactly twice `groups` at every *k* — every group is a
pair, never a larger cluster. The count falls smoothly (5 → 2 → 1 → 1 → 1 → 0) with no
plateau-then-jump, so there is no hidden cluster of near-copies. The zero at *k* = 6 is not an
independent finding: the file stores at most six decimals, so rounding there is equivalent to exact
comparison and only reproduces the deduplication already applied at load time.

**Consistent with repeated records of one company.** Each pair differs in 1 to 6 features out of 64;
the rest match to the last stored digit. In the widest pair all six discrepancies are of the same
relative size (~0.15%) and their signs follow the position of the operating-cost term in each
formula — up where it sits in the numerator (`Attr33`, `Attr34`), down where it sits in the
denominator (`Attr5`, `Attr47`, `Attr52`). A single recomputed cost figure propagating through the
derived ratios produces exactly this; independent noise would not align. `Attr36` is not explained by
this mechanism and is left unexplained rather than rationalised. `Attr29` (logarithm of total assets)
matches in every pair — the only feature that is not scale-invariant, and therefore the only one that
rules out "two structurally similar companies of different size".

**Estimated impact: zero.** None of the ten affected rows is a positive, so no leaked pair can enter
the ranking of positives that PR-AUC depends on. This holds for both split probabilities (0.32 for
the holdout, 0.8 for CV), so the conclusion does not depend on the validation scheme.

**Limitation of the method.** Rounding applies an *absolute* tolerance, so its sensitivity varies
with the magnitude of the column: at *k* = 5 the tolerance is tight for ratios near 1 but far below
the stored precision of `Attr15` (~13740) or `Attr62` (~232), where the comparison stays effectively
exact. The method is therefore **conservative** — it can miss pairs, but it does not manufacture
them. A negative result would have been weak evidence; the five pairs it did find are almost
certainly real.

**Decision: keep these rows.** Five rows out of 5850, flagged by a coarse sieve, with a measured
impact of zero. Validation stays row-wise (`RepeatedStratifiedKFold`); `GroupKFold` is not needed.
The residual risk — pairs this method is blind to — goes to README Limitations.

## 11. Decisions carried forward

| Decision | Evidence |
|---|---|
| Drop duplicate rows at load time, `keep="first"` | 60 groups, 60+ features matching, labels consistent, 4 positives involved |
| Do not drop rows with missing values | 48.8% of rows affected, ~200 of 408 positives would be lost |
| Never fill `Attr37` with 0 or −1 | Missing means "no long-term debt", a state, not a magnitude |
| Linear branch: median imputation **with** indicator | Missingness relates to the target (8.3% vs 6.0%) |
| Boosting branch: leave NaN untouched | LightGBM learns a split direction for missing values natively |
| Candidate feature `has_no_long_term_debt`, to be tested on CV | Possibly redundant for LightGBM given `Attr59` (99.9% agreement) |
| Metrics: PR-AUC and precision@top-k; accuracy excluded | 7.0% positives — a trivial all-negative model reaches 93.0% accuracy |
| Baseline PR-AUC level: **0.070** | Equal to the positive rate; all results reported as lift over it |
| Feature columns keep original `Attr` names | Traceability to the UCI attribute description and published literature |
| Keep near-duplicate rows; validation stays row-wise | 5 pairs at *k* = 1, none positive — estimated PR-AUC shift 0 |
| README limitation: rounding is blind to large-magnitude columns | Absolute tolerance; conservative, so unfound pairs cannot be excluded |
| README limitation: valid only for companies with non-zero revenue | Filter `not ismissing(ATT20)` removed zero-revenue firms |